# Lecture 5


# Lecture Notes: Image Classification with Convolutional Neural Networks (CNNs)

---

## 1. Motivation: Beyond Flat Vectors

### The Problem with Fully Connected (FC) Networks
In standard Multilayer Perceptrons (MLPs) / Fully Connected Layers:
* A 2D image (e.g., $32 \times 32 \times 3$) is flattened into a 1D vector ($3072 \times 1$).
* **This destroys the spatial structure of images.** Pixels next to each other lose their spatial relationship because every pixel is treated as an independent dimension in a giant dot product ($W x$).
* A standard FC layer learns **one global template per class or per hidden neuron**.

```
Flattening: [32 x 32 x 3]  --->  [3072 x 1]  (Spatial relationships lost)
```

---

### Traditional Features vs. End-to-End Deep Learning
Before deep CNNs, computer vision relied on hand-engineered feature representations:
1. **Color Histograms:** Count frequency of colors, ignoring spatial arrangement.
2. **Histogram of Oriented Gradients (HoG):** Count edge directions in local $8 \times 8$ pixel grids.
3. **Bag of Words:** Extract random patches, cluster them into a "codebook" of visual words, and construct a histogram of word counts.

```
[ Image ] ---> [ Hand-crafted Feature Extractor ] ---> [ Linear Classifier ] ---> Scores
```

**The ConvNet Philosophy:**
Instead of fixed, hand-crafted features, train the **feature extractor and classifier together end-to-end** directly from raw pixels.

```
[ Image ] ---> [ Learnable Conv Layers (Feature Extractor) + Classifier ] ---> Scores
```

---

## 2. A Brief History of CNNs

* **Hubel & Wiesel (1959-1968):** Discovered the visual hierarchy in cat visual cortices:
  * **Simple Cells:** Respond to light orientations and edges.
  * **Complex Cells:** Respond to light orientation plus movement.
  * **Hypercomplex Cells:** Respond to movement with an endpoint.
  * **Topographical Mapping:** Nearby cells in the visual cortex represent nearby regions in the visual field.
* **Fukushima (1980) - Neocognitron:** Introduced the "sandwich" architecture alternating Simple ($S$) and Complex ($C$) cell layers. $S$-cells had learnable parameters; $C$-cells performed pooling.
* **LeCun et al. (1998) - LeNet-5:** Applied gradient-based learning (backpropagation) to stacked convolutions and subsampling for digit recognition.
* **Krizhevsky et al. (2012) - AlexNet:** Sparked the modern deep CNN era. Trained a massive network on ImageNet using GPUs, ReLUs, and Dropout.

---

## 3. The Convolutional Layer (CONV)

### Key Intuition & Structure
Unlike FC layers, a **Convolution Layer preserves 3D spatial structure** ($Width \times Height \times Depth$).

* **Filters (Kernels):** Small spatially (e.g., $5 \times 5$ or $3 \times 3$), but **always extend the full depth** of the input volume.
* **Operation:** Slide (convolve) the filter spatially across the input volume, computing a 3D dot product (+ bias) at each location.

$$\text{Output Value} = w^T x + b$$

```
Input Volume:  [32 x 32 x 3]
Filter:        [ 5 x  5 x 3]  (Depth 3 matches input depth 3)
Output:        1 scalar per spatial position -> forms a 2D Activation Map
```

If we use **$K$ different filters**, we get **$K$ activation maps** stacked together to form an output volume of depth $K$.

---

### Spatial Dimension Formulas (The Cheat Sheet)

Given:
* Input size: $W_1 \times H_1 \times C_{in}$
* Filter spatial size: $F \times F$
* Stride: $S$ (step size when sliding)
* Zero Padding: $P$ (border pixels added around input)
* Number of filters: $K$ ($C_{out}$)

#### 1. Output Spatial Dimensions ($W_2 \times H_2 \times K$):
$$W_2 = \frac{W_1 - F + 2P}{S} + 1$$
$$H_2 = \frac{H_1 - F + 2P}{S} + 1$$

> **Note:** If $(W_1 - F + 2P)$ is not evenly divisible by $S$, the stride does not fit properly.

#### 2. Number of Parameters:
$$\text{Parameters per filter} = F \cdot F \cdot C_{in} + 1 \quad (+1 \text{ for bias})$$
$$\text{Total Parameters} = K \cdot (F^2 \cdot C_{in} + 1)$$

---

### Worked Example

* **Input:** $32 \times 32 \times 3$
* **Filters:** $10$ filters of size $5 \times 5$, Stride $S = 1$, Padding $P = 2$

$$\text{Output Width} = \frac{32 - 5 + 2(2)}{1} + 1 = 32$$
$$\text{Output Volume} = 32 \times 32 \times 10$$

$$\text{Params per filter} = (5 \times 5 \times 3) + 1 = 76$$
$$\text{Total Params} = 76 \times 10 = 760 \text{ parameters}$$

---

### Special Case: $1 \times 1$ Convolutions
A $1 \times 1$ filter operates over a single spatial position ($1 \times 1$) across all input channels.
* Works as a **cross-channel linear combination** (a mini FC layer per pixel).
* Used for **dimensionality reduction** (reducing spatial depth/channel size) without altering height or width.


In [ ]:
# Conceptual 1x1 Conv in PyTorch
import torch.nn as nn

# Input: [Batch, 64, 56, 56] -> Output: [Batch, 32, 56, 56]
conv1x1 = nn.Conv2d(in_channels=64, out_channels=32, kernel_size=1, stride=1, padding=0)


---

### Common Conv Hyperparameter Presets
To keep spatial dimensions constant ($W_2 = W_1$), set $S = 1$ and $P = \frac{F - 1}{2}$:
* $F = 3 \implies P = 1$
* $F = 5 \implies P = 2$
* $F = 7 \implies P = 3$
* Number of filters $K$: Powers of 2 (e.g., 32, 64, 128, 256, 512).

---

## 4. Receptive Fields & Downsampling

### Receptive Field
The **receptive field** of an output neuron is the region in the *original input image* that influences that neuron's value.

For $L$ consecutive conv layers with kernel size $K$ and stride $1$:
$$\text{Receptive Field Size} = 1 + L \cdot (K - 1)$$

* *Example:* Three $3 \times 3$ conv layers have a receptive field of $1 + 3(2) = 7 \times 7$.

### Why Downsample?
As images get larger, reaching full image receptive field coverage using only $3 \times 3$ stride-1 convolutions takes **way too many layers**.

**Solutions:**
1. **Strided Convolutions:** Slide filters with stride $S > 1$.
2. **Pooling Layers:** Explicitly downsample spatial dimensions.

---

## 5. Pooling Layer (POOL)

### Max Pooling
Applies a non-linear downsampling operation over depth slices independently.

* **Function:** Selects the maximum activation value in every $F \times F$ window.
* **Purpose:**
  * Shrinks spatial size (reduces computation and memory).
  * Introduces local spatial invariance (small shifts in input do not change max output).
* **Learnable Parameters:** **0** (It uses fixed deterministic math).

```
2x2 Max Pooling with Stride 2:

[ 1  1  2  4 ]          [ Max(1,1,5,6)   Max(2,4,7,8) ]          [ 6  8 ]
[ 5  6  7  8 ]  ----->                                   ----->
[ 3  2  1  0 ]          [ Max(3,2,1,2)   Max(1,0,3,4) ]          [ 3  4 ]
[ 1  2  3  4 ]
```

### Pooling Summary Formulas

* Input: $W_1 \times H_1 \times C$
* Hyperparameters: Spatial extent $F$, Stride $S$

$$W_2 = \frac{W_1 - F}{S} + 1$$
$$H_2 = \frac{H_1 - F}{S} + 1$$
$$\text{Output Volume} = W_2 \times H_2 \times C \quad (\text{Depth } C \text{ is unchanged})$$

**Common Settings:** $F = 2, S = 2$ or $F = 3, S = 2$.

---

## 6. PyTorch Implementation & ConvNet Architecture

### Typical ConvNet Layout
Historically, ConvNets follow a repeating pattern:
`[(CONV -> RELU)*N -> POOL?]*M -> (FC -> RELU)*K -> SOFTMAX`
* $N \approx 1\text{ to }5$, $M$ is large, $K \in [0, 2]$.

*Modern trend:* Prefer smaller filters ($3 \times 3$), ditch POOL/FC layers where possible, and use **strided convolutions** or **Global Average Pooling** instead.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class SimpleConvNet(nn.Module):
    def __init__(self, num_classes=10):
        super(SimpleConvNet, self).__init__()
        # Input: [Batch, 3, 32, 32]

        # Conv Layer 1: preserves size spatially with P=1
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, stride=1, padding=1)

        # Max Pool: reduces size from 32x32 -> 16x16
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

        # Conv Layer 2: preserves size spatially with P=1
        self.conv2 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, stride=1, padding=1)
        # After second max pool, size becomes 8x8

        # Fully Connected Classifier
        self.fc1 = nn.Linear(32 * 8 * 8, 128)
        self.fc2 = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))  # [B, 16, 16, 16]
        x = self.pool(F.relu(self.conv2(x)))  # [B, 32, 8, 8]

        # Flatten for FC layer
        x = x.view(x.size(0), -1)             # [B, 32 * 8 * 8]
        x = F.relu(self.fc1(x))
        x = self.fc2(x)                       # Scores [B, num_classes]
        return x

# Quick instantiation test
model = SimpleConvNet()
dummy_input = torch.randn(2, 3, 32, 32) # Batch of 2 CIFAR-like images
output = model(dummy_input)
print("Output shape:", output.shape)     # torch.Size([2, 10])


---

## 7. What Do Conv Filters Actually Learn?

When visualizing filter weights across layers:
* **First Layer:** Local image templates like **oriented edges**, lines, and opposing color blobs (resembles Gabor filters / V1 visual cortex).
* **Middle Layers:** Complex textures, corners, grid patterns, object parts.
* **Deep Layers:** High-level semantics (faces, dog snouts, wheels, full object templates).

side note: tried rebuilding this part from memory and mixed up the indexing


question to revisit: what happens here when the input size isn't divisible?


this derivation felt shaky, redo it on paper before moving on
